In [1]:
import numpy as py
import pandas as pd

In [2]:
data = [[1, 'Alice'], [2, 'Bob'], [3, 'Tom'], [4, 'Jerry'], [5, 'John']]
customers = pd.DataFrame(data, columns=['customer_id', 'name']).astype({'customer_id':'Int64', 'name':'object'})
data = [[1, '2020-07-31', 1, 1], [2, '2020-7-30', 2, 2], [3, '2020-08-29', 3, 3], [4, '2020-07-29', 4, 1], [5, '2020-06-10', 1, 2], [6, '2020-08-01', 2, 1], [7, '2020-08-01', 3, 3], [8, '2020-08-03', 1, 2], [9, '2020-08-07', 2, 3], [10, '2020-07-15', 1, 2]]
orders = pd.DataFrame(data, columns=['order_id', 'order_date', 'customer_id', 'product_id']).astype({'order_id':'Int64', 'order_date':'datetime64[ns]', 'customer_id':'Int64', 'product_id':'Int64'})
data = [[1, 'keyboard', 120], [2, 'mouse', 80], [3, 'screen', 600], [4, 'hard disk', 450]]
products = pd.DataFrame(data, columns=['product_id', 'product_name', 'price']).astype({'product_id':'Int64', 'product_name':'object', 'price':'Int64'})

In [23]:
df = pd.merge(orders, products, on='product_id')
# count = df.groupby(['customer_id','product_id','product_name']).size().reset_index(name='count').sort_values(by='count',ascending= False)
# df = count.drop_duplicates(subset=['customer_id'])
count = df.groupby(['customer_id','product_id','product_name']).size().reset_index(name='count')
count['rk'] = count.groupby('customer_id')['count'].rank(method='min', ascending=False)
res = count[count['rk']==1][['customer_id','product_id','product_name']]
res

,customer_id,product_id,product_name
1,1,2,mouse
2,2,1,keyboard
3,2,2,mouse
4,2,3,screen
5,3,3,screen
6,4,1,keyboard


In [22]:
data = [[1, 'Moustafa', 100], [2, 'Jonathan', 200], [3, 'Winston', 10000], [4, 'Luis', 800]]
users = pd.DataFrame(data, columns=['user_id', 'user_name', 'credit']).astype({'user_id':'Int64', 'user_name':'object', 'credit':'Int64'})
data = [[1, 1, 3, 400, '2020-08-01'], [2, 3, 2, 500, '2020-08-02'], [3, 2, 1, 200, '2020-08-03']]
transactions = pd.DataFrame(data, columns=['trans_id', 'paid_by', 'paid_to', 'amount', 'transacted_on']).astype({'trans_id':'Int64', 'paid_by':'Int64', 'paid_to':'Int64', 'amount':'Int64', 'transacted_on':'datetime64[ns]'})

In [30]:
pay_out = transactions[['paid_by','amount']].rename(columns={'paid_by':'user_id'})
pay_out['amount'] = -pay_out['amount']
pay_in = transactions[['paid_to','amount']].rename(columns={'paid_to':'user_id'})
all_pay = pd.concat([pay_out, pay_in], ignore_index=True)
user_net = all_pay.groupby('user_id')['amount'].sum().reset_index(name='net')
merge_df = pd.merge(users, user_net, on='user_id', how='right').fillna({'net': 0})
merge_df['credit'] = merge_df['credit'] + merge_df['net']
merge_df['credit_limit_breached']= merge_df['credit'].apply(lambda x: 'Yes' if x < 0 else 'No')
merge_df[['user_id', 'user_name','credit','credit_limit_breached']]

,user_id,user_name,credit,credit_limit_breached
0,1,Moustafa,-100,Yes
1,2,Jonathan,500,No
2,3,Winston,9900,No


In [31]:
data = [[1, 'Winston'], [2, 'Jonathan'], [3, 'Annabelle'], [4, 'Marwan'], [5, 'Khaled']]
customers = pd.DataFrame(data, columns=['customer_id', 'name']).astype({'customer_id':'Int64', 'name':'object'})
data = [[1, '2020-07-31', 1, 1], [2, '2020-7-30', 2, 2], [3, '2020-08-29', 3, 3], [4, '2020-07-29', 4, 1], [5, '2020-06-10', 1, 2], [6, '2020-08-01', 2, 1], [7, '2020-08-01', 3, 1], [8, '2020-08-03', 1, 2], [9, '2020-08-07', 2, 3], [10, '2020-07-15', 1, 2]]
orders = pd.DataFrame(data, columns=['order_id', 'order_date', 'customer_id', 'product_id']).astype({'order_id':'Int64', 'order_date':'datetime64[ns]', 'customer_id':'Int64', 'product_id':'Int64'})
data = [[1, 'keyboard', 120], [2, 'mouse', 80], [3, 'screen', 600], [4, 'hard disk', 450]]
products = pd.DataFrame(data, columns=['product_id', 'product_name', 'price']).astype({'product_id':'Int64', 'product_name':'object', 'price':'Int64'})

In [32]:
max_date = orders.groupby("product_id")["order_date"].max().reset_index()
latest_order = pd.merge(orders, max_date, on=["product_id", "order_date"])
res = pd.merge(products, latest_order, on="product_id", how="left")
res[["product_id", "product_name", "order_date"]]

,product_id,product_name,order_date
0,1,keyboard,2020-08-01
1,1,keyboard,2020-08-01
2,2,mouse,2020-08-03
3,3,screen,2020-08-29
4,4,hard disk,NaT
